# Can the video give us a better ball? (experiment)

**For whoever runs this:** you need the repository (`git pull`), the match data folder with
`tactical/BAR-ATH_tactical.mp4` in it, and ideally a GPU (Apple Silicon `mps` or CUDA). Run the
cells from top to bottom. Steps 1–5 take about **15–30 minutes** and tell us whether the idea works
at all. Step 6, the full run, is only worth starting if step 5 says *go*.

**Why we are trying this.** Our error analysis (`notebooks/possession_walkthrough.ipynb`, section 8)
says almost all the remaining error is in **short possessions**, and that they are limited by the
tracking's ball: smoothed over about 0.8 s, so a touch is blurred, and missing on 41% of graded
frames. A ball detected frame by frame in the video could be **sharper** and fill some **gaps**.

**How.** A detector pretrained on everyday photos (YOLO, COCO class "sports ball") looks at a
640×640 crop around where the tracking says the ball is. If the tracking has no ball, it looks
around the last detection; if there is none, it searches the whole frame every 5th frame. The
code is in `src/possession/video.py`.

**Rules.** The detector is pretrained on other images and nothing here uses a label of the match,
so this is allowed. Its output is derived match data: it stays in the data folder and **never goes
into git**.

**What to send back** (section 7 says how):
1. the numbers printed in steps 5 and 7 (`experiments/video_report.json`; fine to commit);
2. if you ran step 6, the file `.cache/video_ball_tactical.npz` from the data folder (a few MB),
   sent privately (AirDrop, Drive), **not** through GitHub.

# Setup

In [ ]:
# Run once. Installs the detector and the video reader (PyTorch comes with ultralytics).
%pip install -q "ultralytics>=8.1" opencv-python pandas matplotlib

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

# Where the data folder is (it contains tactical/). Order: environment, then .env, then this default.
DEFAULT_DATA_DIR = "/Users/mvrm/Desktop/Hackathons/barca/data"
if "MTAH_DATA_DIR" not in os.environ:
    env = ROOT / ".env"
    if env.exists():
        for line in env.read_text().splitlines():
            if line.startswith("MTAH_DATA_DIR="):
                os.environ["MTAH_DATA_DIR"] = line.split("=", 1)[1].strip().strip('"')
    os.environ.setdefault("MTAH_DATA_DIR", DEFAULT_DATA_DIR)
DATA_DIR = Path(os.environ["MTAH_DATA_DIR"])
VIDEO = DATA_DIR / "tactical" / "BAR-ATH_tactical.mp4"
OUT = DATA_DIR / ".cache" / "video_ball_tactical.npz"          # derived data: never commit
print("data folder:", DATA_DIR, "| video found:", VIDEO.exists())
assert VIDEO.exists(), "set MTAH_DATA_DIR (or DEFAULT_DATA_DIR above) to the folder that contains tactical/"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

import games
from possession import video as vid
from possession.atd import load_atd
from possession.homography import load_homography

# Settings you may change
WEIGHTS = "yolov8m.pt"      # downloaded automatically; "yolov8x.pt" is slower but better on small objects
CONF = 0.05                 # keep weak candidates: the ball is tiny, the selection step filters
BATCH = 32                  # crops per detector call; lower it if memory runs out
DEVICE = vid.best_device()  # "mps" on an M1/M2/M3 Mac
print("device:", DEVICE)

# 1. Is it the right video?

Row *i* of the submission is frame *i* of the **tactical** video: 199,319 frames, 25 fps,
1920×1080. If these numbers differ, stop and tell us. (The frame count reported by a video library
can be off by a few frames; a big difference means the wrong file, for example the broadcast one.)

In [ ]:
info = vid.video_info(VIDEO)
print(info)
W, H_PX = info["width"], info["height"]
if abs(info["frames"] - 199319) > 5 or round(info["fps"]) != 25 or (W, H_PX) != (1920, 1080):
    print("WARNING: this does not look like the tactical video described in DATA.md")

# 2. Tracking and camera

We load the tracking (positions in metres) and the homography (one 3×3 matrix per frame, mapping
pitch coordinates 0–1 to pixels). Projecting the tracked ball through the homography tells us where
in the image the ball should be, frame by frame.

In [ ]:
atd = load_atd("tactical")
Hm, reproj_err, valid = load_homography("tactical")
pitch = np.array(atd.pitch)
period = np.zeros(atd.n_frames, np.int8)
for half, first, last in games.half_video_blocks(games.GAME):
    period[first:last + 1] = half
graded = period > 0
expected_px = vid.project(Hm, atd.ball_xy / pitch)           # NaN where the tracking has no ball
inside = np.isfinite(expected_px[:, 0]) & graded
print(f"tracking ball present on {inside.sum() / graded.sum():.1%} of graded frames")
print(f"near-player share of the tracking ball: "
      f"{vid.near_player_share(atd.ball_xy, atd.players_xy, inside):.3f}  (clean public tracking: about 0.41)")

# 3. Are the frames aligned?

Two checks, because everything else depends on them.

**a. Seeking.** Jumping to a frame in a compressed video can land on the wrong frame with some
codecs. We read the same frame twice, once by jumping straight to it and once by jumping 300 frames
earlier and reading forward. The two images must be identical.

In [ ]:
for f in (20000, 60000, 130000):
    a = next(vid.read_frames(VIDEO, [f], seek_gap=0))[1]
    b = [img for g, img in vid.read_frames(VIDEO, [f - 300, f], seek_gap=250) if g == f][0]
    print(f"frame {f}: identical = {np.array_equal(a, b)}")

**b. Does the tracked ball sit on the ball in the image?** The red circle is the tracking's ball
projected into the frame. Look at the zoomed views: the circle should be on the ball, or close to it
when the ball is moving fast (the tracking is smoothed). A circle that is consistently somewhere
else means the video is not the one the tracking was made from.

In [ ]:
def show_frames(frames_, marks, zoom=160, title=""):
    fig, axes = plt.subplots(2, len(frames_), figsize=(4.2 * len(frames_), 6))
    imgs = dict(vid.read_frames(VIDEO, sorted(frames_)))
    for k, f in enumerate(frames_):
        img = cv2.cvtColor(imgs[f], cv2.COLOR_BGR2RGB)
        axes[0, k].imshow(img)
        axes[0, k].set_title(f"frame {f}")
        u, v = expected_px[f] if np.isfinite(expected_px[f]).all() else (W / 2, H_PX / 2)
        axes[1, k].imshow(img)
        axes[1, k].set_xlim(u - zoom, u + zoom)
        axes[1, k].set_ylim(v + zoom, v - zoom)
        for color, uv in marks(f):
            for ax in axes[:, k]:
                ax.scatter(*uv, s=250 if ax is axes[1, k] else 60, facecolor="none", edgecolor=color, lw=2)
        for ax in axes[:, k]:
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

rng = np.random.default_rng(0)
check_frames = sorted(rng.choice(np.flatnonzero(inside), 4, replace=False))
show_frames(check_frames, lambda f: [("red", expected_px[f])], title="red = tracking ball")

# 4. The detector on a few frames

Same frames, now with the detector's choice in **green**. The detector runs on a 640×640 crop
around the red circle.

In [ ]:
detector = vid.yolo_detector(WEIGHTS, device=DEVICE, conf=CONF)
st_demo = vid.detect(vid.read_frames(VIDEO, check_frames), (W, H_PX), expected_px, detector, batch=BATCH)
for f in check_frames:
    print(f"frame {f}: detected = {np.isfinite(st_demo['uv'][f, 0])}, confidence {st_demo['conf'][f]:.2f}")
show_frames(check_frames, lambda f: [("red", expected_px[f]), ("lime", st_demo["uv"][f])],
            title="red = tracking ball, green = detected ball")

# 5. The real test: 12 clips of 20 seconds

12 clips spread evenly over both halves, 6,000 frames in total. For each frame we keep the detected
ball, then compare it with the tracking **without any label**:

| question | number to look at | **go** if |
|---|---|---|
| does it find the ball, not heads or shoes? | distance video vs tracking, median / share > 5 m | < 1 m / < 10% |
| does it see the ball when the tracking does? | detection rate where the tracking has a ball | > 70% |
| is it **sharper** than the tracking? | player within 1 m: video vs tracking ball | video ≥ tracking + 0.05 |
| does it **fill gaps** in live play? | detection rate where the tracking has none and players move | > 30% |
| is a full run affordable? | projected full-run time | < 8 h |

The first two say whether the detector works; the last three say whether it can help. Sharpness is
the most valuable: the gaps during live play are only about 6% of graded frames.

In [ ]:
CLIPS, CLIP_LEN = 12, 20 * 25
starts = []
for half, first, last in games.half_video_blocks(games.GAME):
    starts += list(np.linspace(first + 2000, last - 2000 - CLIP_LEN, CLIPS // 2).astype(int))
sample = np.concatenate([np.arange(s, s + CLIP_LEN) for s in starts])
in_sample = np.zeros(atd.n_frames, bool)
in_sample[sample] = True

t0 = time.time()
st = vid.new_state(atd.n_frames)
for s in starts:                                    # one clip at a time, so tracking resets between clips
    vid.detect(vid.read_frames(VIDEO, np.arange(s, s + CLIP_LEN)), (W, H_PX), expected_px, detector,
               state=st, batch=BATCH)
elapsed = time.time() - t0
fps = len(sample) / elapsed
print(f"{len(sample)} frames in {elapsed / 60:.1f} min: {fps:.1f} frames/s on {DEVICE}")
print(f"projected full run over the 158,857 graded frames: {158857 / fps / 3600:.1f} h")

sample_report = vid.report(st, Hm, atd.ball_xy, atd.pitch, period, atd.players_speed, atd.players_xy,
                           frames=in_sample)
sample_report["frames per second"] = fps
sample_report["device"], sample_report["weights"], sample_report["conf"] = DEVICE, WEIGHTS, CONF
for k, v in sample_report.items():
    print(f"  {k:<85} {round(v, 3) if isinstance(v, float) else v}")

**One more alignment check.** If the video and the tracking are aligned, the two balls should be
closest at a time shift of 0 frames (or shifted by a constant 1–3 frames if the tracking's smoothing
delays it a little). A minimum far from 0 means a frame offset: tell us.

In [ ]:
xy_video = vid.unproject(Hm, st["uv"].astype(np.float64)) * pitch
for lag in range(-6, 7, 2):
    f = sample[(sample + lag >= 0) & (sample + lag < atd.n_frames)]
    d = np.hypot(*(xy_video[f] - atd.ball_xy[f + lag]).T)
    print(f"tracking shifted by {lag:+d} frames: median distance {np.nanmedian(d):.2f} m")

In [ ]:
# extra code – trajectories of one clip, tracking vs video, on the pitch
s = starts[2]
seg = np.arange(s, s + CLIP_LEN)
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.plot(*atd.ball_xy[seg].T, "-", color="#ff7f0e", lw=2, label="tracking ball")
ax.plot(*xy_video[seg].T, ".", color="k", ms=3, label="video ball")
ax.set_xlim(0, pitch[0]); ax.set_ylim(0, pitch[1]); ax.set_aspect("equal")
ax.set_title(f"frames {seg[0]}–{seg[-1]}: corners should be sharper in black than in orange", loc="left")
ax.legend()
plt.show()

In [ ]:
REPORT = ROOT / "experiments" / "video_report.json"
REPORT.parent.mkdir(exist_ok=True)
REPORT.write_text(json.dumps({"sample (12 x 20 s)": sample_report}, indent=1))
print("wrote", REPORT.relative_to(ROOT), "(numbers only, fine to commit)")

**Stop here if** the median distance is above about 2 m or the detection rate where the tracking
has a ball is below 50%: the detector is not finding the ball. Things worth one more try before
giving up: `WEIGHTS = "yolov8x.pt"` (bigger model), or `CONF = 0.15` (fewer false detections). Re-run
from step 4 and send us both reports.

Otherwise, go on with the full run.

# 6. Full run (only if step 5 says go)

This processes every graded frame and **saves after every chunk**, so it can be stopped and
restarted at any time: it picks up where it stopped. Keep the Mac plugged in, and stop it from
sleeping by running `caffeinate -dims` in a Terminal window while this runs.

In [ ]:
CHUNK = 5000
st_full = dict(np.load(OUT)) if OUT.exists() else vid.new_state(atd.n_frames)
todo = np.flatnonzero(graded & ~st_full["done"])
print(f"{(graded & st_full['done']).sum():,} graded frames already done, {len(todo):,} to go")

timer = vid.Timer()
for i in range(0, len(todo), CHUNK):
    chunk = todo[i:i + CHUNK]
    vid.detect(vid.read_frames(VIDEO, chunk), (W, H_PX), expected_px, detector, state=st_full, batch=BATCH)
    OUT.parent.mkdir(exist_ok=True)
    np.savez_compressed(OUT, **st_full)
    rate, eta = timer.eta(i + len(chunk), len(todo))
    print(f"{i + len(chunk):>7,}/{len(todo):,} frames · {rate:.1f} frames/s · about {eta / 60:.0f} min left", flush=True)
print("done; saved", OUT)

# 7. Final report, and what to send back

In [ ]:
full_report = vid.report(st_full, Hm, atd.ball_xy, atd.pitch, period, atd.players_speed, atd.players_xy)
for k, v in full_report.items():
    print(f"  {k:<85} {round(v, 3) if isinstance(v, float) else v}")
saved = json.loads(REPORT.read_text()) if REPORT.exists() else {}
saved["full run"] = full_report
REPORT.write_text(json.dumps(saved, indent=1))
print("\nupdated", REPORT.relative_to(ROOT))

**Send back:**

1. `experiments/video_report.json`: commit and push it, or paste it in the chat. It only contains
   aggregate numbers.
2. `video_ball_tactical.npz` from the data folder's `.cache/`: **privately** (AirDrop, Drive...), never
   through GitHub. It is derived match data, and `.gitignore` already keeps it out of git.

We will then rebuild the ball features from the video ball, recalibrate the training simulation to
it, and compare against the current submission on held-out public matches.